# Day 071 — Exercise 1: cosine_similarity + ImageIndex

**What you'll build:** The retrieval core of Vision RAG — `cosine_similarity` and `ImageIndex` with `add`, `search`, and `__len__`.

**Why it matters:** Everything else in Vision RAG feeds data into these two pieces. Getting them right means the search results are correct.

In [ ]:
import numpy as np


## Task

**`cosine_similarity(a, b) -> float`:**
- Convert both to `np.float32` arrays
- `denom = np.linalg.norm(va) * np.linalg.norm(vb)`
- Return `0.0` if `denom == 0.0`, else `float(np.dot(va, vb) / denom)`

**`ImageIndex`:**
- `__init__`: `self._items = []`
- `add`: append `{id, description, embedding (as np.float32 array), metadata}`
- `search`: score all items with `cosine_similarity`, sort descending, return top-n as dicts with keys `id, description, score, metadata` (no embedding)
- `__len__`: return `len(self._items)`

## Your Implementation

In [ ]:
def cosine_similarity(a, b) -> float:
    """Cosine similarity between two vectors.

    Returns 0.0 if either vector has zero norm.
    """
    raise NotImplementedError


class ImageIndex:
    """In-memory image search index using cosine similarity."""

    def __init__(self) -> None:
        raise NotImplementedError

    def add(self, image_id: str, description: str, embedding,
            metadata=None) -> None:
        """Add an image to the index."""
        raise NotImplementedError

    def search(self, query_embedding, n: int = 5) -> list:
        """Return top-n results sorted by cosine similarity (descending).

        Each result dict: {id, description, score, metadata}.
        """
        raise NotImplementedError

    def __len__(self) -> int:
        raise NotImplementedError


In [ ]:
def cosine_similarity(a, b) -> float:
    va = np.array(a, dtype=np.float32)
    vb = np.array(b, dtype=np.float32)
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom == 0.0:
        return 0.0
    return float(np.dot(va, vb) / denom)


class ImageIndex:
    def __init__(self):
        self._items = []

    def add(self, image_id, description, embedding, metadata=None):
        self._items.append({
            'id':          image_id,
            'description': description,
            'embedding':   np.array(embedding, dtype=np.float32),
            'metadata':    metadata or {},
        })

    def search(self, query_embedding, n=5):
        if not self._items:
            return []
        q = np.array(query_embedding, dtype=np.float32)
        scored = [(cosine_similarity(q, item['embedding']), item)
                  for item in self._items]
        scored.sort(key=lambda x: -x[0])
        top = scored[:min(n, len(scored))]
        return [
            {'id':          item['id'],
             'description': item['description'],
             'score':       float(score),
             'metadata':    item['metadata']}
            for score, item in top
        ]

    def __len__(self):
        return len(self._items)


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # cosine_similarity: identical vectors = 1.0
    sim = cosine_similarity([1, 0, 0], [1, 0, 0])
    assert abs(sim - 1.0) < 1e-5, f"Expected 1.0, got {sim}"
    score += 1; print("\u2705 cosine_similarity: identical vectors = 1.0")

    # cosine_similarity: orthogonal = 0.0
    sim2 = cosine_similarity([1, 0], [0, 1])
    assert abs(sim2) < 1e-5, f"Expected 0.0, got {sim2}"
    score += 1; print("\u2705 cosine_similarity: orthogonal vectors = 0.0")

    # cosine_similarity: zero vector → 0.0
    sim3 = cosine_similarity([0, 0, 0], [1, 2, 3])
    assert sim3 == 0.0
    score += 1; print("\u2705 cosine_similarity: zero vector returns 0.0")

    # ImageIndex: add and len
    idx = ImageIndex()
    idx.add('a', 'red car', [1.0, 0.0], {'tag': 'car'})
    idx.add('b', 'blue sky', [0.0, 1.0], {'tag': 'sky'})
    assert len(idx) == 2
    score += 1; print("\u2705 ImageIndex: add + __len__")

    # search: sorted by score desc, correct keys
    results = idx.search([1.0, 0.0], n=2)
    assert len(results) == 2
    assert results[0]['id'] == 'a', f"Expected 'a', got {results[0]['id']}"
    assert results[0]['score'] > results[1]['score']
    assert 'description' in results[0] and 'metadata' in results[0]
    score += 1; print("\u2705 search: sorted descending, correct keys")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def cosine_similarity(a, b) -> float:
    va = np.array(a, dtype=np.float32)
    vb = np.array(b, dtype=np.float32)
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom == 0.0:
        return 0.0
    return float(np.dot(va, vb) / denom)


class ImageIndex:
    def __init__(self):
        self._items = []

    def add(self, image_id, description, embedding, metadata=None):
        self._items.append({
            'id':          image_id,
            'description': description,
            'embedding':   np.array(embedding, dtype=np.float32),
            'metadata':    metadata or {},
        })

    def search(self, query_embedding, n=5):
        if not self._items:
            return []
        q = np.array(query_embedding, dtype=np.float32)
        scored = [(cosine_similarity(q, item['embedding']), item)
                  for item in self._items]
        scored.sort(key=lambda x: -x[0])
        top = scored[:min(n, len(scored))]
        return [
            {'id':          item['id'],
             'description': item['description'],
             'score':       float(score),
             'metadata':    item['metadata']}
            for score, item in top
        ]

    def __len__(self):
        return len(self._items)
```

**Why `np.float32` not `float64`?** float32 halves memory usage with negligible precision loss for cosine similarity. Embedding models typically output float32.

**Why exclude `embedding` from results?** A 768-dim embedding is ~3 KB per result — 10 results would add 30 KB to every search response. Callers never need the raw embedding back.

</details>